## Tools: Giving CineBot Hands

## 2.1 The Problem: A Model That Can't Check Anything Real

CineBot can understand a booking request perfectly now. But ask it something that requires
checking a real, live fact:


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Missing OPENAI_API_KEY -- check your .env file or Colab Secrets"

from langchain.chat_models import init_chat_model
model = init_chat_model("openai:gpt-5-nano")
print("Environment ready. CineBot's brain (the model) is connected.")


In [ ]:
response = model.invoke("Is Interstellar showing at 7pm tonight at the Downtown cinema?")
print(response.content)


In [ ]:
from langchain_core.tools import tool

@tool
def check_showtimes(movie_title: str) -> str:
    """Check available showtimes for a movie at the cinema.

    Args:
        movie_title: The exact title of the movie to check
    """
    fake_showtimes = {
        "interstellar": "7:00 PM and 10:15 PM",
        "dune part two": "9:30 PM only",
        "oppenheimer": "Sold out for tonight",
    }
    return fake_showtimes.get(movie_title.lower(), "No showtimes found for that title.")

print("Tool name:       ", check_showtimes.name)
print("Tool description:", check_showtimes.description)
print("Tool args schema:", check_showtimes.args)


The docstring's `Args:` section matters, not just decoration — it sharpens what the model
understands about each individual parameter, worth writing for any tool with more than one
argument.

### Overriding the Name and Description


In [ ]:
@tool("book_seats", description="Book cinema seats for a customer. Use this whenever a customer wants to reserve tickets.")
def reserve(movie: str, seats: int) -> str:
    """Reserve seats."""  # this docstring is now IGNORED -- description= above wins
    return f"Reserved {seats} seat(s) for {movie}"

print("Description actually shown to the model:", reserve.description)


## 2.3 You Don't Always Need to Write a Tool Yourself: Prebuilt Tools

Before writing a custom tool for something common — like web search — check if a ready-made
package already exists. **Tavily** is a real, widely-used search tool built specifically for AI
agents, available as a LangChain-compatible tool with almost no setup.


In [ ]:
# pip install langchain-tavily  (requires a free Tavily API key -- tavily.com)
#
# from langchain_tavily import TavilySearch
#
# search_tool = TavilySearch(max_results=3)
# # or wrapped with a custom name/description, exactly like any other tool:
# @tool("search_internet_with_tavily", description="Use this when the user wants to search the internet.")
# def search_internet(query: str) -> str:
#     """Search the internet for current information."""
#     return search_tool.invoke(query)

print("TavilySearch is a real, documented prebuilt tool -- not something you build from scratch.")
print("Not installed in this notebook's environment (needs its own API key) -- pattern shown")
print("here for reference; install and configure it yourself before running this cell for real.")


## 2.4 Advanced Input Schemas: `args_schema`

For genuinely complex inputs, define the schema explicitly with a Pydantic model instead of
relying on inline type hints alone — richer per-field guidance, same idea as Chapter 1.


In [ ]:
class SeatBookingInput(BaseModel):
    """Input for booking cinema seats."""
    movie_title: str = Field(description="Exact movie title")
    seat_count: int = Field(description="Number of seats to book", ge=1, le=10)
    preferred_row: Literal["front", "middle", "back"] = Field(default="middle", description="Preferred seating row")

@tool(args_schema=SeatBookingInput)
def book_seats(movie_title: str, seat_count: int, preferred_row: str = "middle") -> str:
    """Book seats for a movie."""
    return f"Booked {seat_count} seat(s) in the {preferred_row} row for {movie_title}."

print(book_seats.args)


## 2.5 The Problem: A Name Collision You Won't See Coming

Two argument names are reserved by the framework and will break your tool if reused:
`config` and `runtime`.


In [ ]:
try:
    @tool
    def broken_tool(movie_title: str, config: str) -> str:
        """A tool that accidentally uses a reserved argument name."""
        return f"{movie_title}, {config}"
    broken_tool.invoke({"movie_title": "Dune", "config": "test"})
except Exception as e:
    print(f"Failed -- 'config' is reserved: {type(e).__name__}: {e}")

print()
print("Fix: just pick any other name.")

@tool
def fixed_tool(movie_title: str, settings: str) -> str:
    """A tool using a safe, non-reserved argument name."""
    return f"{movie_title}, {settings}"
print(fixed_tool.invoke({"movie_title": "Dune", "settings": "test"}))


## 2.6 The Problem: Does Binding a Tool Mean It Runs?

`.bind_tools()` only makes the model AWARE tools exist — it does not run anything.


In [ ]:
model_with_tools = model.bind_tools([check_showtimes, book_seats])
response = model_with_tools.invoke("Is Interstellar show available tonight?")

print("Tool calls REQUESTED (nothing has executed yet):")
for tool_call in response.tool_calls:
    print("-->", tool_call["name"], tool_call["args"])


```mermaid
graph LR
    A[Model + Tools] -->|.bind_tools| B[model_with_tools]
    B -->|.invoke question| C{Model decides}
    C -->|needs a tool| D["tool_calls populated -- REQUEST only"]
    D -.nothing executes yet.-> E[create_agent runs it]
```

Actually running the tool and looping back for a final answer is `create_agent`'s job — the
same pattern from Chapter 1's agent-level structured output.


## 2.7 The Problem: A Tool That Needs to See the Conversation Itself

Suppose a tool needs to know the last thing the customer mentioned — information already sitting
in the conversation, but not something you'd want to pass in as a manual argument (the model
would have to already know the answer to hand it in).

## 2.8 The Fix: `ToolRuntime` — a Two-Way Mirror

**Analogy:** the model sees only its own reflection (the tool's declared arguments). A tool that
accepts a special `runtime: ToolRuntime` parameter can see through to a back room full of
resources — and the model never even knows that back room exists.

```mermaid
graph LR
    A[Model's View] -->|sees only declared args| B[Mirror]
    B -.hidden back room, tool-only.-> C[runtime.state - this conversation]
    B -.-> D[runtime.context - per-run config]
    B -.-> E[runtime.store - long-term memory]
```


In [ ]:
from langchain.tools import tool, ToolRuntime
from langchain_core.messages import HumanMessage

@tool
def get_last_movie_mentioned(runtime: ToolRuntime) -> str:
    """Find the last movie title the customer mentioned in this conversation."""
    for message in reversed(runtime.state["messages"]):
        if isinstance(message, HumanMessage):
            return f"Last thing the customer said: {message.content}"
    return "No customer messages found."

# Prove the hiding claim directly -- 'runtime' will NOT appear here:
print("Tool schema seen by the model:", get_last_movie_mentioned.args)


### ToolRuntime: provides below Runtime information
1. State
2. Context
3. Store
4. Stream Writer
5. Execution Info
6. Server Info
7. Config: RunnableConfig
8. Tool Call ID

`runtime` genuinely isn't in that schema. LangChain automatically excludes it because it
recognizes the `ToolRuntime` type annotation — the model cannot see this parameter exists.

### `runtime.store`: Remembering a Customer Across Entirely Separate Visits

`runtime.state` only covers THIS conversation. For memory that survives across completely
separate sessions, a `Store` is attached to the agent and reached via `runtime.store`.


In [ ]:
from typing import Any
from langgraph.store.memory import InMemoryStore

loyalty_store = InMemoryStore()

@tool
def save_favourite_genre(customer_id: str, genre: str, runtime: ToolRuntime) -> str:
    """Save a customer's favorite movie genre for future visits."""
    runtime.store.put((customer_id, "preferences"), "favorite_genre", {"value": genre})
    return f"Got it -- I'll remember you like {genre} movies."

@tool
def recall_favorite_genre(customer_id: str, runtime: ToolRuntime) -> str:
    """Recall a customer's favorite movie genre, if we've saved it before."""
    result = runtime.store.get((customer_id, "preferences"), "favorite_genre")
    return result.value["value"] if result else "We don't have a saved preference for this customer yet."

from langchain.agents import create_agent
memory_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[save_favourite_genre, recall_favorite_genre],
    store=loyalty_store,
)

memory_agent.invoke({"messages": [("user", "Hi, I'm customer priya_01, I love sci-fi movies, please remember that.")]})
result = memory_agent.invoke({"messages": [("user", "What genre do I usually like? I'm priya_01.")]})
print(result["messages"][-1].content)


### Inspecting the Store Directly: `.search()`

Beyond `.get()` (fetch one specific key) and `.put()` (save one), a `Store` also supports
`.search()` — list every item saved under a given namespace, without needing to know each exact
key in advance. Genuinely useful for debugging or building an admin view of what's been saved.


In [ ]:
items = loyalty_store.search(("priya_01", "preferences"))
for item in items:
    print(item)


### Two Lesser-Known `ToolRuntime` Components

`execution_info` gives the current thread/run/retry identity. `server_info` is populated ONLY
when running on LangGraph Server — `None` locally, which is expected, not a bug.


In [ ]:
@tool
def log_booking_context(runtime: ToolRuntime) -> str:
    """Log identity info about the current booking session -- useful for debugging."""
    info = runtime.execution_info
    print(f"  [debug] thread={info.thread_id}, run={info.run_id}, attempt={info.node_attempt}")
    server = runtime.server_info
    print(f"  [debug] server_info is None locally: {server is None}")
    return "Logged."

print("Tool defined. server_info being None locally is EXPECTED --")
print("it only populates once deployed to LangGraph Server, never during local development.")


## 2.9 Skipping the Model's Final Polish: `return_direct`

Sometimes a tool's raw output IS the final answer — no rephrasing needed. Use when the exact
wording matters, like a legally-worded policy that must never be paraphrased.


In [ ]:
@tool(return_direct=True)
def get_exact_refund_policy() -> str:
    """Tell the refund policy."""
    return "Tickets are refundable up to 2 hours before showtime. No refunds after that."

direct_agent = create_agent(model="openai:gpt-5-mini", tools=[get_exact_refund_policy])
result = direct_agent.invoke({"messages": [("user", "What's your refund policy?")]})
print(result["messages"][-1].content)


If we have hundreds of tools and each tool has information of ~500 token, so showing all available tools to LLM will sent unnecessary tool tokens also. 
In this case ~500 * 500 tools = 2,50,000 Total tokens for tools information only will be sent to LLM in each user query.
Another problem is our Model will also get confused.

<img src="../../assets/many_tools.png" width="400" height="400">

The exact string from the tool, verbatim — not a model-rephrased version of it.

## 2.10 The Problem: Should Every Customer See Every Tool?

CineBot has a VIP lounge booking option. A naive fix is telling the model in the system prompt
"don't offer this to non-VIP customers" — but that relies on the model choosing to follow an
instruction. A clever or confused customer could still talk it into trying. We want something
genuinely stronger.

## 2.11 The Fix: `wrap_model_call` — Making a Tool Genuinely Disappear

**Analogy — a menu that reprints itself before you sit down:** a VIP member sees the full menu.
A regular guest sees a shorter menu — not because they're told "please don't order the VIP
items," but because those items genuinely aren't printed on their menu at all.


In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@tool
def standard_booking(movie_title: str) -> str:
    """Book a standard seat."""
    return f"Standard seat booked for {movie_title}."

@tool
def vip_lounge_booking(movie_title: str) -> str:
    """Book a VIP lounge seat with premium service. VIP members only."""
    return f"VIP lounge seat booked for {movie_title}."

@wrap_model_call
def gate_vip_tools(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Only expose vip_lounge_booking to VIP members."""
    is_vip = request.state.get("is_vip_member", False)
    if not is_vip:
        allowed = [t for t in request.tools if t.name != "vip_lounge_booking"]
        request = request.override(tools=allowed)
    return handler(request)

gated_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[standard_booking, vip_lounge_booking],
    middleware=[gate_vip_tools],
)

result_regular = gated_agent.invoke({"messages": [("user", "Book me a VIP lounge seat for Dune?")]})
print("Regular member result:", result_regular["messages"][-1].content)


In [ ]:
result_vip = gated_agent.invoke(
    {"messages": [("user", "Book me a VIP lounge seat for Dune")], "is_vip_member": True}
)
print("VIP member result:", result_vip["messages"][-1].content)
print()
print("Same code, same query -- only the 'is_vip_member' flag differed. The model literally")
print("could not choose vip_lounge_booking in the first case -- it wasn't on its menu at all.")


<img src="../../assets/tool_runtime_information.png" width="800" height="400">

## Chapter 2 Summary

- A tool is a phone line out of an otherwise stuck model; the docstring is its entire pitch for
  when to use it.
- Check for a **prebuilt tool** (like Tavily search) before writing one from scratch.
- `args_schema` handles complex inputs; `config` and `runtime` are reserved argument names.
- `.bind_tools()` only makes a model aware of tools — `create_agent` actually runs them.
- `ToolRuntime` gives hidden access to `state`, `context`, `store` (including `.search()` to
  list saved items), `execution_info`, and `server_info` — invisible to the model.
- `return_direct=True` skips the model's final rephrasing pass.
- Tools can be filtered out of a model's awareness entirely, based on runtime conditions — a
  genuinely stronger guarantee than instructing the model not to use something.
